In [ ]:
import polars as pl

In [ ]:
number_work_days = 220
number_of_occurrences_per_day = number_work_days // 5


(
    pl.scan_csv("../data/summary.csv")
    .with_columns(
        pl.col("datetime").cast(pl.Datetime),
        pl.col("from").cast(pl.Categorical),
        pl.col("to").cast(pl.Categorical),
    )
    .sort("datetime", "from", "to", descending=True)
    .drop("from_to", "duration_s")
    .select("datetime", "from", "to", "distance_m", "duration_in_traffic_s")
    .with_columns(duration_at_50kph_s=pl.col("distance_m") / 1000 / 50 * 3600)
    .with_columns(
        delay_s=pl.col("duration_in_traffic_s") - pl.col("duration_at_50kph_s")
    )
    .with_columns(
        day_of_week_n=pl.col("datetime").dt.weekday(),
        day_of_week=pl.col("datetime").dt.to_string("%A"),
    )
    .with_columns(hour=pl.col("datetime").dt.hour())
    .with_columns(qoi=pl.col("delay_s"))
    .group_by("day_of_week_n", "from", "to", "hour", maintain_order=True)
    .agg(
        week_day=pl.col("day_of_week").unique().first(),
        kph50=pl.col("duration_at_50kph_s").first(),
        min=pl.col("qoi").min(),
        q05=pl.col("qoi").quantile(0.05),
        mean=pl.col("qoi").mean(),
        median=pl.col("qoi").median(),
        q95=pl.col("qoi").quantile(0.95),
        max=pl.col("qoi").max(),
        std=pl.col("qoi").std(),
    )
    .filter(pl.col("week_day").is_in(["Saturday", "Sunday"]).not_())
    .filter(
        (
            (
                (
                    (pl.col("hour") == 7)
                    & (pl.col("from") == "Krankenhaus")
                    & (pl.col("to") == "Höhle")
                )
                | (
                    (pl.col("hour") == 16)
                    & (pl.col("from") == "Höhle")
                    & (pl.col("to") == "Krankenhaus")
                )
            )
            & (pl.col("week_day") != "Tuesday")
        )
        | (
            (
                (
                    (pl.col("hour") == 6)
                    & (pl.col("from") == "Krankenhaus")
                    & (pl.col("to") == "Höhle")
                )
                | (
                    (pl.col("hour") == 15)
                    & (pl.col("from") == "Höhle")
                    & (pl.col("to") == "Krankenhaus")
                )
            )
            & (pl.col("week_day") == "Tuesday")
        )
    )
    .sort("day_of_week_n", "hour")
    .drop("from", "to", "hour", "week_day", "day_of_week_n")
    .sum()
    .with_columns((pl.all() * number_of_occurrences_per_day / 3600).round(1))
    .collect()
    # .to_pandas()
    # .style.format(precision=1)
    # .background_gradient(subset=["mean"], cmap="Reds", vmin=0)
)